In [116]:
import pandas as pd

We utilize this notebook as a logical follow-up step to the *Data Wrangling* folder's two ipynb files. In order to understand the meaning of the MERGE1 csv file and the subsequent work that is performed in this notebook, we suggest reviewing the previous steps in that folder.

We begin our workflow here by reading out the MERGE1 csv we downloaded at the end of our workflow in the *Data Wrangling* folder.

In [117]:
merge1 = pd.read_csv("csv_data/MERGE1.csv")
print(merge1.columns)

Index(['costat', 'curcd', 'datafmt', 'indfmt', 'consol', 'sic', 'datadate',
       'gvkey', 'conm', 'tic', 'fyear', 'at', 'ceq', 'dltt', 'lse', 'ni',
       'revt', 'xrd', 'csho', 'prcc_f', 'sich', 'mkt_cap', 'industry',
       'boardid', 'companyid', 'datestartrole', 'directorid', 'directorname',
       'companyname', 'rolename', 'dateendrole', 'datestartroleflag',
       'dateendroleflag', 'seniority', 'UG', 'top20_ug', 'MBA', 'top20_mba',
       'PhD', 'top20_phd', 'MD', 'top20_md', 'Master's', 'top20_masters',
       'dob', 'gender', 'diversitynetworklabel'],
      dtype='object')


Before proceeding with any further data refinement, it is important that we ensure we are not working with any data corrupted by duplicate entries for a given firm+year combo. We check this in the following way:

In [118]:
dups = merge1[merge1.duplicated(subset=['gvkey', 'fyear'], keep=False)]
len(dups)

100

This means we have around 100 duplications at play. Let's check why these might be happening

In [119]:
print(dups[['gvkey', 'fyear', 'directorid', 'directorname', 'rolename']].sort_values(['gvkey', 'fyear']))

      gvkey  fyear  directorid            directorname                rolename
125    8020   2021    732483.0  Doctor Quinton Hennigh  Co-Chairman/Acting CEO
126    8020   2021   1264832.0          Rob Humphryson                     CEO
220   12142   2015     86007.0               Mark Hurd                  Co-CEO
221   12142   2015     33503.0              Safra Catz                  Co-CEO
222   12142   2016     86007.0               Mark Hurd                  Co-CEO
..      ...    ...         ...                     ...                     ...
949  162335   2022   1376569.0       George Paleologou           President/CEO
950  162335   2023   1094011.0            Ron Lombardi  Chairman/President/CEO
951  162335   2023   1376569.0       George Paleologou           President/CEO
952  162335   2024   1094011.0            Ron Lombardi  Chairman/President/CEO
953  162335   2024   1376569.0       George Paleologou           President/CEO

[100 rows x 5 columns]


It appears that we are facing situations where there are some co-CEOs, so we will eliminate the problem by picking the "higher seniority" CEO for each case of a duplicate. We sort by seniority (high to low) and keep the first of all duplicates on the axes of gvkey and fyear.

In [120]:
# eliminate co-CEOs by picking higher seniority
merge1 = merge1.sort_values('seniority', ascending=False)\
               .drop_duplicates(subset=['gvkey', 'fyear'], keep='first')

In [121]:
print(merge1[['gvkey', 'fyear']].duplicated().sum())
print(len(merge1))

0
1032


Performing our redundancy reduction eliminated all redundancy issues in our dataset. We now have unique CEO+firm+year combos for all of our compustat data entries. Let's check what NaN values we have left...

In [122]:
merge1.isna().sum()

costat                     0
curcd                      0
datafmt                    0
indfmt                     0
consol                     0
sic                        0
datadate                   0
gvkey                      0
conm                       0
tic                        0
fyear                      0
at                         0
ceq                        0
dltt                       0
lse                        0
ni                         0
revt                       0
xrd                       87
csho                       0
prcc_f                     0
sich                       3
mkt_cap                    0
industry                   0
boardid                    0
companyid                  0
datestartrole              0
directorid                 0
directorname               0
companyname                0
rolename                   0
dateendrole                0
datestartroleflag          0
dateendroleflag            0
seniority                  0
UG            

Our only NaN values are missing R&D entries (expected for some companies that perhaps don't track R&D) and missing educational qualifications for 45 CEOs (potentials for dropping in our regression environment). Let's save down our more cleaned data!

In [123]:
merge1.shape
merge1.to_csv("csv_data/MERGE2.csv", index=False)

In [124]:
# we try to get our directorid column into a string for input into boardex
%run processes/directorid_txt_conversion.py

Next, we are interested in actually gathering some compensation data for a more refined control variable set. To do so, we are going to be accessing execucomp, which has robust data about CEO compenstiaon packages. However, this requires us to perform a manual bridge from BoardEx CEO ids to compustat CEO ids. We use the above-generated txt file to download this bridge and bring in that csv, comparing the initial number of unique CEOs in our dataset to our other dataset.

In [125]:
len(merge1["directorid"].unique().tolist())

259

In [137]:
# we download our bridge between boardex and execucomp
bridge = pd.read_csv("csv_data/boardex_to_execucomp.csv")

In [138]:
len(bridge)

214

In [139]:
# we just want our CEOs from 2015-2025 in a txt file with execucomp credentials
%run processes/execid_txt_conversion.py

With our updated execucomp credentials, we perform a datapull with basic compensation data from execucomp. We download our csv and migrate this into our workflow below.

In [140]:
execucomp = pd.read_csv("csv_data/execucomp_compensation.csv")
execucomp.isna().sum()

execid                        0
year                          0
gvkey                         0
exec_fullname                 0
co_per_rol                    0
bonus                         0
eip_unearn_val               12
option_awards_blk_value    1047
opt_unex_exer_est_val        12
opt_unex_unexer_est_val      12
salary                        0
shrown_excl_opts             14
stock_awards_fv              12
tdc1                         12
dtype: int64

We have no blk scholes option data so we delete this field. We also compute cash comp in total and pct_equity (stock / total comp) as additional fields added to our dataset.

In [141]:
execucomp = execucomp.drop(columns=['option_awards_blk_value'])
execucomp = execucomp.dropna()

execucomp['cash_comp'] = execucomp['salary'] + execucomp['bonus']
execucomp['pct_equity'] = execucomp['stock_awards_fv'] / execucomp['tdc1']

In [142]:
# we have some duplicates in our bridge and we drop them for our merge. we then merge the datasets
bridge_clean = bridge.drop_duplicates(subset=['directorid'], keep='first')

merge2 = merge1.merge(
    bridge_clean[['directorid', 'execid']],
    on='directorid',
    how='left'
)
print(merge2.shape)

(1032, 48)


In [143]:
merge1.shape

(1032, 47)

Now we finally perform our merge with our execucomp dataset, adding the salary figures from execucomp and expanding our dataset.

In [144]:
merge3 = merge2.merge(
    execucomp[['execid', 'year', 'salary', 'bonus', 'tdc1', 
                'stock_awards_fv', 'cash_comp', 'pct_equity', 'opt_unex_exer_est_val', 'opt_unex_unexer_est_val']],
    left_on=['execid', 'fyear'],
    right_on=['execid', 'year'],
    how='left'
)

In [145]:
print(merge3.shape)
print(merge3[['gvkey', 'fyear']].duplicated().sum())

(1034, 57)
2


In [146]:
merge3.isna().sum()

costat                       0
curcd                        0
datafmt                      0
indfmt                       0
consol                       0
sic                          0
datadate                     0
gvkey                        0
conm                         0
tic                          0
fyear                        0
at                           0
ceq                          0
dltt                         0
lse                          0
ni                           0
revt                         0
xrd                         87
csho                         0
prcc_f                       0
sich                         3
mkt_cap                      0
industry                     0
boardid                      0
companyid                    0
datestartrole                0
directorid                   0
directorname                 0
companyname                  0
rolename                     0
dateendrole                  0
datestartroleflag            0
dateendr

In [147]:
merge3.to_csv("csv_data/MERGE3.csv", index=False)

With our 3 basic merge files performed, we are ready to prep our data for some regressions and carry out the analysis portion of this thesis. To follow along, proceed to the *Regressions* folder and start with *merge2_prep.ipynb*